# Model Training and Evaluation

These notes document the saved experiments in their original order. All validation results use one stratified 80/20 split; F1 refers to class +1. Threshold searches use this same validation set, so their selected scores are development estimates, not independent test results.

## 1. Data Loading


In [3]:
import numpy as np

from helpers import load_csv_data

In [4]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(
    "data/dataset"
)

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)

x_train: (328135, 321)
y_train: (328135,)
x_test : (109379, 321)


## 2. Stratified Training/Validation Split

Split each class separately using seed 42 to preserve the imbalanced label distribution. The official test inputs are kept separate from these experiments.


In [5]:
def stratified_train_val_split(x, y, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)

    train_indices = []
    val_indices = []

    for label in np.unique(y):
        indices = np.where(y == label)[0]
        rng.shuffle(indices)

        n_val = int(len(indices) * val_ratio)

        val_indices.append(indices[:n_val])
        train_indices.append(indices[n_val:])

    train_indices = np.concatenate(train_indices)
    val_indices = np.concatenate(val_indices)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)

    return (
        x[train_indices],
        x[val_indices],
        y[train_indices],
        y[val_indices],
    )

In [6]:
x_tr, x_val, y_tr, y_val = stratified_train_val_split(
    x_train,
    y_train,
    val_ratio=0.2,
    seed=42,
)

In [7]:
print("Training set:")
print("  X shape:", x_tr.shape)
print("  y shape:", y_tr.shape)

print("\nValidation set:")
print("  X shape:", x_val.shape)
print("  y shape:", y_val.shape)

for name, y in [
    ("Original", y_train),
    ("Train", y_tr),
    ("Validation", y_val),
]:
    positive_rate = np.mean(y == 1)
    print(f"{name:10s} positive rate: {positive_rate:.2%}")

Training set:
  X shape: (262508, 321)
  y shape: (262508,)

Validation set:
  X shape: (65627, 321)
  y shape: (65627,)
Original   positive rate: 8.83%
Train      positive rate: 8.83%
Validation positive rate: 8.83%


The split contains 262,508 training and 65,627 validation samples. Both retain an 8.83% positive rate, matching the original labels.


### Median Imputation

Estimate column medians on the internal training split and reuse them for validation. The saved checks show no undefined medians and no remaining NaN values after imputation.


In [8]:
feature_medians = np.nanmedian(x_tr, axis=0)

print("Number of NaN medians:", np.isnan(feature_medians).sum())

Number of NaN medians: 0


In [9]:
x_tr_imp = np.where(
    np.isnan(x_tr),
    feature_medians,
    x_tr,
)

x_val_imp = np.where(
    np.isnan(x_val),
    feature_medians,
    x_val,
)

print("NaNs in training:", np.isnan(x_tr_imp).sum())
print("NaNs in validation:", np.isnan(x_val_imp).sum())

NaNs in training: 0
NaNs in validation: 0


### Standardization

Fit means and standard deviations on imputed training data only. Six zero-variance columns use a scale of 1 to avoid division by zero. The resulting matrices contain no NaN or Inf values; constant training columns remain zero.


In [10]:
feature_means = np.mean(x_tr_imp, axis=0)
feature_stds = np.std(x_tr_imp, axis=0)

print("Zero-std features:", np.sum(feature_stds == 0))

Zero-std features: 6


In [11]:
safe_stds = feature_stds.copy()
safe_stds[safe_stds == 0] = 1.0

x_tr_std = (x_tr_imp - feature_means) / safe_stds
x_val_std = (x_val_imp - feature_means) / safe_stds

In [12]:
print("Training NaNs:", np.isnan(x_tr_std).sum())
print("Validation NaNs:", np.isnan(x_val_std).sum())

print("Training Infs:", np.isinf(x_tr_std).sum())
print("Validation Infs:", np.isinf(x_val_std).sum())

print("Mean of standardized training features:",
      np.mean(x_tr_std, axis=0)[:10])

print("Std of standardized training features:",
      np.std(x_tr_std, axis=0)[:10])

Training NaNs: 0
Validation NaNs: 0
Training Infs: 0
Validation Infs: 0
Mean of standardized training features: [ 2.01478403e-16 -4.02688246e-16 -6.00085805e-17  9.20915679e-17
 -3.44216167e-16 -6.61417675e-13 -2.48396732e-15 -8.70565832e-12
 -8.70565832e-12  0.00000000e+00]
Std of standardized training features: [1. 1. 1. 1. 1. 1. 1. 1. 1. 0.]


## 3. Ridge Regression and Decision Thresholds

Fit Ridge with lambda = 0.01 on standardized features and labels in {-1, +1}. This initial run has no intercept; its scores are not probabilities.


Ridge Regression baseline

In [13]:
from implementations import ridge_regression

In [14]:
lambda_ = 0.01

w, loss = ridge_regression(
    y_tr,
    x_tr_std,
    lambda_,
)

print("Weights shape:", w.shape)
print("Training loss:", loss)

Weights shape: (321,)
Training loss: 0.47586081454757856


In [15]:
val_scores = x_val_std @ w

y_val_pred = np.where(val_scores >= 0, 1, -1)

print("Predicted -1:", np.sum(y_val_pred == -1))
print("Predicted +1:", np.sum(y_val_pred == 1))

Predicted -1: 37028
Predicted +1: 28599


In [16]:
tp = np.sum((y_val == 1) & (y_val_pred == 1))
tn = np.sum((y_val == -1) & (y_val_pred == -1))
fp = np.sum((y_val == -1) & (y_val_pred == 1))
fn = np.sum((y_val == 1) & (y_val_pred == -1))

accuracy = (tp + tn) / len(y_val)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

print(f"TP: {tp}")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")

print(f"\nAccuracy : {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall   : {recall:.2%}")
print(f"F1 score : {f1:.4f}")

TP: 5213
TN: 36446
FP: 23386
FN: 582

Accuracy : 63.48%
Precision: 18.23%
Recall   : 89.96%
F1 score : 0.3031


At threshold 0, the model predicts 28,599 positives: recall is 89.96%, but precision is 18.23% and accuracy is 63.48%. This motivates inspecting score separation and changing the decision threshold rather than rejecting the model solely on accuracy.


In [17]:
print("Validation score statistics:")
print(f"Min   : {np.min(val_scores):.4f}")
print(f"25%   : {np.percentile(val_scores, 25):.4f}")
print(f"Median: {np.median(val_scores):.4f}")
print(f"75%   : {np.percentile(val_scores, 75):.4f}")
print(f"Max   : {np.max(val_scores):.4f}")

print("\nBy true class:")
print(f"Mean score y=-1: {np.mean(val_scores[y_val == -1]):.4f}")
print(f"Mean score y=+1: {np.mean(val_scores[y_val == 1]):.4f}")

Validation score statistics:
Min   : -3.0269
25%   : -0.1692
Median: -0.0401
75%   : 0.1378
Max   : 1.4179

By true class:
Mean score y=-1: -0.0261
Mean score y=+1: 0.2705


In [18]:
def evaluate_predictions(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return accuracy, precision, recall, f1

### Coarse Threshold Search

Evaluate thresholds from 0 to 0.50 in steps of 0.05 using the same scores. Higher thresholds trade recall for precision; the reported optimum is only among these candidates.


In [19]:
thresholds = np.arange(0.0, 0.51, 0.05)

print(
    f"{'Threshold':>9} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds:
    y_pred = np.where(val_scores >= threshold, 1, -1)

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:9.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

Threshold   Accuracy  Precision     Recall         F1    Pred +1
     0.00     63.48%     18.23%     89.96%     0.3031      28599
     0.05     69.97%     20.80%     85.54%     0.3346      23830
     0.10     75.64%     23.73%     79.41%     0.3654      19395
     0.15     80.15%     26.69%     71.46%     0.3886      15515
     0.20     83.75%     30.03%     63.18%     0.4071      12192
     0.25     86.49%     33.44%     53.51%     0.4116       9274
     0.30     88.48%     37.10%     43.74%     0.4015       6833
     0.35     89.84%     41.03%     34.44%     0.3745       4865
     0.40     90.65%     44.95%     26.40%     0.3326       3404
     0.45     91.12%     49.29%     19.79%     0.2824       2327
     0.50     91.32%     53.15%     14.25%     0.2248       1554


The highest recorded F1 is 0.4116 at threshold 0.25 (accuracy 86.49%). Threshold 0.50 achieves the highest grid accuracy, 91.32%, but recall drops to 14.25% and F1 to 0.2248.


### Majority-Class Reference

Predict -1 for every validation sample to establish a reference that does not use features.


In [20]:
majority_pred = np.full_like(y_val, -1)

acc, precision, recall, f1 = evaluate_predictions(
    y_val, majority_pred
)

print(f"Majority baseline accuracy : {acc:.2%}")
print(f"Majority baseline precision: {precision:.2%}")
print(f"Majority baseline recall   : {recall:.2%}")
print(f"Majority baseline F1       : {f1:.4f}")

Majority baseline accuracy : 91.17%
Majority baseline precision: 0.00%
Majority baseline recall   : 0.00%
Majority baseline F1       : 0.0000


The majority baseline reaches 91.17% accuracy but zero positive-class F1. Thus, high accuracy alone can conceal poor positive-class detection.


## 4. Logistic Regression: Intercept and Thresholds

Convert training labels to {0, 1} for logistic loss. First use the same standardized features without an intercept, with 100 iterations and gamma = 0.1.


Logistic Regression baseline

In [21]:
y_tr_log = (y_tr == 1).astype(float)

print(np.unique(y_tr_log, return_counts=True))

(array([0., 1.]), array([239328,  23180]))


In [22]:
from implementations import logistic_regression

initial_w = np.zeros(x_tr_std.shape[1])

w_log, loss_log = logistic_regression(
    y_tr_log,
    x_tr_std,
    initial_w,
    max_iters=100,
    gamma=0.1,
)

print("Weights shape:", w_log.shape)
print("Training logistic loss:", loss_log)

Weights shape: (321,)
Training logistic loss: 0.6688728090830308


In [23]:
val_logits = x_val_std @ w_log

val_probs = 1.0 / (1.0 + np.exp(-np.clip(val_logits, -500, 500)))

print("Probability statistics:")
print(f"Min   : {np.min(val_probs):.4f}")
print(f"Median: {np.median(val_probs):.4f}")
print(f"Mean  : {np.mean(val_probs):.4f}")
print(f"Max   : {np.max(val_probs):.4f}")

Probability statistics:
Min   : 0.0003
Median: 0.4794
Mean  : 0.4988
Max   : 0.9599


In [24]:
y_val_pred_log = np.where(val_probs >= 0.5, 1, -1)

accuracy, precision, recall, f1 = evaluate_predictions(
    y_val,
    y_val_pred_log,
)

print("Predicted -1:", np.sum(y_val_pred_log == -1))
print("Predicted +1:", np.sum(y_val_pred_log == 1))

print(f"\nAccuracy : {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall   : {recall:.2%}")
print(f"F1 score : {f1:.4f}")

Predicted -1: 37054
Predicted +1: 28573

Accuracy : 63.46%
Precision: 18.18%
Recall   : 89.65%
F1 score : 0.3023


Without an intercept, training loss is 0.6689 and mean predicted probability is 0.4988, far above the observed positive rate. At threshold 0.5, F1 is 0.3023. The next experiment adds a free offset to represent the class imbalance.


### Add an Intercept

Append a column of ones and repeat the 100-iteration fit at the same learning rate. This isolates the intercept change within the logistic experiments.


In [25]:
# Add intercept term

x_tr_int = np.column_stack([
    np.ones(x_tr_std.shape[0]),
    x_tr_std
])

x_val_int = np.column_stack([
    np.ones(x_val_std.shape[0]),
    x_val_std
])

print("Training shape:", x_tr_int.shape)
print("Validation shape:", x_val_int.shape)

Training shape: (262508, 322)
Validation shape: (65627, 322)


In [26]:
initial_w_int = np.zeros(x_tr_int.shape[1])

w_log_int, loss_log_int = logistic_regression(
    y_tr_log,
    x_tr_int,
    initial_w_int,
    max_iters=100,
    gamma=0.1,
)

print("Weights shape:", w_log_int.shape)
print("Intercept:", w_log_int[0])
print("Training logistic loss:", loss_log_int)

Weights shape: (322,)
Intercept: -1.7794282593758057
Training logistic loss: 0.2714666213430098


In [27]:
val_logits_int = x_val_int @ w_log_int

val_probs_int = 1.0 / (
    1.0 + np.exp(-np.clip(val_logits_int, -500, 500))
)

print("Probability statistics:")
print(f"Min   : {np.min(val_probs_int):.4f}")
print(f"Median: {np.median(val_probs_int):.4f}")
print(f"Mean  : {np.mean(val_probs_int):.4f}")
print(f"Max   : {np.max(val_probs_int):.4f}")

Probability statistics:
Min   : 0.0000
Median: 0.1334
Mean  : 0.1649
Max   : 0.8905


In [28]:
y_val_pred_log_int = np.where(
    val_probs_int >= 0.5,
    1,
    -1
)

accuracy, precision, recall, f1 = evaluate_predictions(
    y_val,
    y_val_pred_log_int,
)

print("Predicted -1:", np.sum(y_val_pred_log_int == -1))
print("Predicted +1:", np.sum(y_val_pred_log_int == 1))

print(f"\nAccuracy : {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall   : {recall:.2%}")
print(f"F1 score : {f1:.4f}")

Predicted -1: 64965
Predicted +1: 662

Accuracy : 91.36%
Precision: 59.37%
Recall   : 6.78%
F1 score : 0.1217


In [29]:
thresholds_log = np.arange(0.05, 0.51, 0.05)

print(
    f"{'Threshold':>9} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds_log:
    y_pred = np.where(
        val_probs_int >= threshold,
        1,
        -1
    )

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:9.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

Threshold   Accuracy  Precision     Recall         F1    Pred +1
     0.05      9.62%      8.89%     99.83%     0.1632      65091
     0.10     41.58%     12.88%     97.48%     0.2276      43842
     0.15     63.45%     18.23%     90.04%     0.3032      28629
     0.20     76.38%     23.98%     77.15%     0.3659      18645
     0.25     83.99%     30.01%     61.00%     0.4023      11781
     0.30     88.19%     36.25%     44.52%     0.3996       7118
     0.35     90.21%     42.38%     30.22%     0.3528       4132
     0.40     91.10%     48.95%     19.79%     0.2819       2343
     0.45     91.34%     54.36%     12.15%     0.1986       1295
     0.50     91.36%     59.37%      6.78%     0.1217        662


With an intercept, training loss falls to 0.2715. Threshold 0.5 yields 91.36% accuracy but only 6.78% recall. The coarse search instead selects threshold 0.25 for F1 = 0.4023 (accuracy 83.99%).


## 5. Missing-Indicator Experiment

Append one binary NaN indicator per original feature before adding the intercept, producing 643 columns. Keep median-imputed, standardized values and train for 100 iterations at gamma = 0.1.


Missing Indicators experiment

In [30]:
train_missing = np.isnan(x_tr).astype(float)
val_missing = np.isnan(x_val).astype(float)

print("Train missing indicators:", train_missing.shape)
print("Validation missing indicators:", val_missing.shape)

print("Indicator values:", np.unique(train_missing))

Train missing indicators: (262508, 321)
Validation missing indicators: (65627, 321)
Indicator values: [0. 1.]


In [31]:
x_tr_miss = np.column_stack([
    x_tr_std,
    train_missing
])

x_val_miss = np.column_stack([
    x_val_std,
    val_missing
])

print("Training shape:", x_tr_miss.shape)
print("Validation shape:", x_val_miss.shape)

print("Training NaNs:", np.isnan(x_tr_miss).sum())
print("Validation NaNs:", np.isnan(x_val_miss).sum())

Training shape: (262508, 642)
Validation shape: (65627, 642)
Training NaNs: 0
Validation NaNs: 0


In [32]:
x_tr_miss_int = np.column_stack([
    np.ones(x_tr_miss.shape[0]),
    x_tr_miss
])

x_val_miss_int = np.column_stack([
    np.ones(x_val_miss.shape[0]),
    x_val_miss
])

print("Training shape with intercept:", x_tr_miss_int.shape)
print("Validation shape with intercept:", x_val_miss_int.shape)

Training shape with intercept: (262508, 643)
Validation shape with intercept: (65627, 643)


In [33]:
initial_w_miss = np.zeros(x_tr_miss_int.shape[1])

w_log_miss, loss_log_miss = logistic_regression(
    y_tr_log,
    x_tr_miss_int,
    initial_w_miss,
    max_iters=100,
    gamma=0.1,
)

print("Weights shape:", w_log_miss.shape)
print("Intercept:", w_log_miss[0])
print("Training logistic loss:", loss_log_miss)

Weights shape: (643,)
Intercept: -0.026556679266000683
Training logistic loss: 0.23061230309035596


In [34]:
val_logits_miss = x_val_miss_int @ w_log_miss

val_probs_miss = 1.0 / (
    1.0 + np.exp(-np.clip(val_logits_miss, -500, 500))
)

print("Probability statistics:")
print(f"Min   : {np.min(val_probs_miss):.4f}")
print(f"Median: {np.median(val_probs_miss):.4f}")
print(f"Mean  : {np.mean(val_probs_miss):.4f}")
print(f"Max   : {np.max(val_probs_miss):.4f}")

Probability statistics:
Min   : 0.0000
Median: 0.0434
Mean  : 0.0887
Max   : 0.9323


In [35]:
y_val_pred_miss = np.where(
    val_probs_miss >= 0.5,
    1,
    -1
)

accuracy, precision, recall, f1 = evaluate_predictions(
    y_val,
    y_val_pred_miss
)

print("Predicted -1:", np.sum(y_val_pred_miss == -1))
print("Predicted +1:", np.sum(y_val_pred_miss == 1))

print(f"\nAccuracy : {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall   : {recall:.2%}")
print(f"F1 score : {f1:.4f}")

Predicted -1: 64831
Predicted +1: 796

Accuracy : 91.36%
Precision: 57.79%
Recall   : 7.94%
F1 score : 0.1396


In [36]:
thresholds_miss = np.arange(0.05, 0.51, 0.05)

print(
    f"{'Threshold':>9} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds_miss:
    y_pred = np.where(
        val_probs_miss >= threshold,
        1,
        -1
    )

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:9.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

Threshold   Accuracy  Precision     Recall         F1    Pred +1
     0.05     60.90%     17.38%     91.30%     0.2920      30444
     0.10     75.81%     23.57%     77.53%     0.3614      19066
     0.15     83.19%     29.20%     63.43%     0.4000      12587
     0.20     86.93%     33.65%     49.37%     0.4002       8503
     0.25     89.07%     38.14%     38.14%     0.3814       5795
     0.30     90.19%     41.79%     28.39%     0.3381       3936
     0.35     90.88%     46.53%     21.74%     0.2964       2708
     0.40     91.21%     50.65%     16.10%     0.2443       1842
     0.45     91.31%     53.76%     11.23%     0.1858       1211
     0.50     91.36%     57.79%      7.94%     0.1396        796


Training loss falls to 0.2306, but the highest grid F1 is 0.4002 at threshold 0.20, slightly below 0.4023 without indicators. This run does not demonstrate a validation benefit; it also does not rule one out, since optimization was limited to 100 iterations.


## 6. Logistic Convergence Analysis

Return to the intercept-only design without missing indicators and record training loss. History entries are measured before each update, unlike a returned loss evaluated at final weights.


In [38]:
def logistic_loss_history(y, tx, initial_w, max_iters, gamma):
    w = initial_w.copy()
    losses = []

    for _ in range(max_iters):
        scores = tx @ w
        probs = 1.0 / (1.0 + np.exp(-np.clip(scores, -500, 500)))

        # Binary cross-entropy
        eps = 1e-15
        loss = -np.mean(
            y * np.log(probs + eps)
            + (1 - y) * np.log(1 - probs + eps)
        )
        losses.append(loss)

        # Gradient
        gradient = tx.T @ (probs - y) / len(y)

        # GD update
        w = w - gamma * gradient

    return w, np.array(losses)

In [39]:
w_conv, losses_conv = logistic_loss_history(
    y_tr_log,
    x_tr_int,
    np.zeros(x_tr_int.shape[1]),
    max_iters=100,
    gamma=0.1,
)

for i in [0, 9, 19, 49, 99]:
    print(f"Iteration {i + 1:3d}: loss = {losses_conv[i]:.6f}")

Iteration   1: loss = 0.693147
Iteration  10: loss = 0.548975
Iteration  20: loss = 0.458208
Iteration  50: loss = 0.333811
Iteration 100: loss = 0.272094


Loss decreases from 0.693147 to 0.272094 over the first 100 recorded iterations. Extend the run to check whether the earlier baseline was limited by optimization time.


In [40]:
w_conv_500, losses_conv_500 = logistic_loss_history(
    y_tr_log,
    x_tr_int,
    np.zeros(x_tr_int.shape[1]),
    max_iters=500,
    gamma=0.1,
)

for i in [99, 199, 299, 399, 499]:
    print(f"Iteration {i + 1:3d}: loss = {losses_conv_500[i]:.6f}")

Iteration 100: loss = 0.272094
Iteration 200: loss = 0.242096
Iteration 300: loss = 0.233842
Iteration 400: loss = 0.230390
Iteration 500: loss = 0.228619


In [41]:
val_logits_500 = x_val_int @ w_conv_500

val_probs_500 = 1.0 / (
    1.0 + np.exp(-np.clip(val_logits_500, -500, 500))
)

print("Probability statistics:")
print(f"Min   : {np.min(val_probs_500):.4f}")
print(f"Median: {np.median(val_probs_500):.4f}")
print(f"Mean  : {np.mean(val_probs_500):.4f}")
print(f"Max   : {np.max(val_probs_500):.4f}")

Probability statistics:
Min   : 0.0000
Median: 0.0512
Mean  : 0.0980
Max   : 0.9763


In [42]:
thresholds_500 = np.arange(0.05, 0.51, 0.05)

print(
    f"{'Threshold':>9} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds_500:
    y_pred = np.where(val_probs_500 >= threshold, 1, -1)

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:9.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

Threshold   Accuracy  Precision     Recall         F1    Pred +1
     0.05     57.06%     16.32%     93.58%     0.2779      33232
     0.10     73.78%     22.71%     81.95%     0.3557      20910
     0.15     81.74%     28.12%     68.61%     0.3989      14138
     0.20     86.20%     33.23%     55.82%     0.4166       9734
     0.25     88.71%     37.94%     43.80%     0.4066       6690
     0.30     90.12%     42.47%     33.56%     0.3749       4580
     0.35     90.88%     46.95%     25.49%     0.3304       3146
     0.40     91.25%     51.17%     19.28%     0.2800       2183
     0.45     91.40%     55.25%     13.99%     0.2233       1468
     0.50     91.47%     60.12%     10.09%     0.1729        973


At 500 iterations, the recorded loss reaches 0.228619 and the best grid F1 increases to 0.4166 at threshold 0.20. Further loss reduction supports continuing the optimization analysis, without establishing exact convergence.


## 7. Learning-Rate Tuning

Compare gamma values 0.05, 0.10, 0.20, and 0.50 with the same 500-iteration budget. This measures convergence speed and validation performance within the tested range.


In [43]:
gammas = [0.05, 0.2, 0.5]

gamma_results = {}

for gamma in gammas:
    w_g, losses_g = logistic_loss_history(
        y_tr_log,
        x_tr_int,
        np.zeros(x_tr_int.shape[1]),
        max_iters=500,
        gamma=gamma,
    )

    gamma_results[gamma] = (w_g, losses_g)

    print(
        f"gamma={gamma:.2f} | "
        f"loss100={losses_g[99]:.6f} | "
        f"loss500={losses_g[499]:.6f}"
    )

gamma=0.05 | loss100=0.333259 | loss500=0.236973
gamma=0.20 | loss100=0.242069 | loss500=0.225986
gamma=0.50 | loss100=0.228586 | loss500=0.225196


In [44]:
all_models = {
    0.05: gamma_results[0.05][0],
    0.10: w_conv_500,
    0.20: gamma_results[0.2][0],
    0.50: gamma_results[0.5][0],
}

thresholds = np.arange(0.05, 0.51, 0.05)

print(
    f"{'Gamma':>6} "
    f"{'Best threshold':>14} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'Best F1':>10}"
)

for gamma, w in all_models.items():

    logits = x_val_int @ w
    probs = 1.0 / (
        1.0 + np.exp(-np.clip(logits, -500, 500))
    )

    best = None

    for threshold in thresholds:
        y_pred = np.where(probs >= threshold, 1, -1)

        accuracy, precision, recall, f1 = evaluate_predictions(
            y_val, y_pred
        )

        if best is None or f1 > best[-1]:
            best = (
                threshold,
                accuracy,
                precision,
                recall,
                f1,
            )

    threshold, accuracy, precision, recall, f1 = best

    print(
        f"{gamma:6.2f} "
        f"{threshold:14.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f}"
    )

 Gamma Best threshold   Accuracy  Precision     Recall    Best F1
  0.05           0.20     84.76%     31.28%     60.64%     0.4127
  0.10           0.20     86.20%     33.23%     55.82%     0.4166
  0.20           0.20     86.68%     34.13%     54.63%     0.4201
  0.50           0.20     86.80%     34.31%     54.15%     0.4201


Gamma 0.20 and 0.50 both give F1 = 0.4201 at threshold 0.20, at the displayed precision. Gamma 0.50 reaches the lowest recorded training loss in this comparison and is carried forward; this is not a claim that it is universally optimal.


## 8. Regularized Logistic Regression

Using the intercept design, 500 iterations, and gamma = 0.50, compare L2 strengths from 1e-5 to 0.1 against the unregularized model. Inspect weight norms and validation results; reported training loss is separate from the regularization penalty.


Regularized Logistic Regression

In [45]:
from implementations import reg_logistic_regression

lambdas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

reg_models = {}

for lambda_ in lambdas:
    w_reg, loss_reg = reg_logistic_regression(
        y_tr_log,
        x_tr_int,
        lambda_,
        np.zeros(x_tr_int.shape[1]),
        max_iters=500,
        gamma=0.5,
    )

    reg_models[lambda_] = w_reg

    print(
        f"lambda={lambda_:>7g} | "
        f"training loss={loss_reg:.6f} | "
        f"weight norm={np.linalg.norm(w_reg):.4f}"
    )

lambda=  1e-05 | training loss=0.225198 | weight norm=3.4198
lambda= 0.0001 | training loss=0.225232 | weight norm=3.3877
lambda=  0.001 | training loss=0.226216 | weight norm=3.1293
lambda=   0.01 | training loss=0.246317 | weight norm=2.2245
lambda=    0.1 | training loss=0.384473 | weight norm=0.9999


In [46]:
all_reg_models = {
    0.0: gamma_results[0.5][0],
    **reg_models,
}

thresholds = np.arange(0.05, 0.51, 0.05)

print(
    f"{'Lambda':>10} "
    f"{'Best threshold':>14} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'Best F1':>10}"
)

for lambda_, w in all_reg_models.items():

    logits = x_val_int @ w
    probs = 1.0 / (
        1.0 + np.exp(-np.clip(logits, -500, 500))
    )

    best = None

    for threshold in thresholds:
        y_pred = np.where(probs >= threshold, 1, -1)

        accuracy, precision, recall, f1 = evaluate_predictions(
            y_val, y_pred
        )

        if best is None or f1 > best[-1]:
            best = (
                threshold,
                accuracy,
                precision,
                recall,
                f1,
            )

    threshold, accuracy, precision, recall, f1 = best

    print(
        f"{lambda_:10g} "
        f"{threshold:14.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f}"
    )

    Lambda Best threshold   Accuracy  Precision     Recall    Best F1
         0           0.20     86.80%     34.31%     54.15%     0.4201
     1e-05           0.20     86.79%     34.30%     54.15%     0.4200
    0.0001           0.20     86.75%     34.23%     54.32%     0.4200
     0.001           0.20     86.43%     33.80%     55.98%     0.4215
      0.01           0.25     87.00%     34.44%     52.27%     0.4152
       0.1           0.35     79.42%     24.07%     61.78%     0.3465


Lambda = 0.001 gives the highest recorded grid F1, 0.4215 at threshold 0.20, versus 0.4201 without regularization. Stronger regularization shrinks weights but reduces F1. The small improvement on one reused validation split is not evidence of a robust generalization gain.


## 9. Least Squares Baseline

Fit Least Squares with an intercept on {-1, +1} labels. Its training MSE is 0.136248. Inspect score ranges before searching thresholds, since these regression scores have a different scale from logistic probabilities.


In [47]:
from implementations import least_squares

w_ls, loss_ls = least_squares(
    y_tr,
    x_tr_int
)

print("Weights shape:", w_ls.shape)
print("Training loss:", loss_ls)
print("Weight norm:", np.linalg.norm(w_ls))

Weights shape: (322,)
Training loss: 0.1362479199820299
Weight norm: 4.446818307719441


In [48]:
val_scores_ls = x_val_int @ w_ls

print("Validation score statistics:")
print(f"Min   : {np.min(val_scores_ls):.4f}")
print(f"25%   : {np.percentile(val_scores_ls, 25):.4f}")
print(f"Median: {np.median(val_scores_ls):.4f}")
print(f"75%   : {np.percentile(val_scores_ls, 75):.4f}")
print(f"Max   : {np.max(val_scores_ls):.4f}")

print("\nBy true class:")
print(f"Mean score y=-1: {np.mean(val_scores_ls[y_val == -1]):.4f}")
print(f"Mean score y=+1: {np.mean(val_scores_ls[y_val == 1]):.4f}")

Validation score statistics:
Min   : -3.8866
25%   : -0.9963
Median: -0.8693
75%   : -0.6848
Max   : 0.6313

By true class:
Mean score y=-1: -0.8505
Mean score y=+1: -0.5424


In [49]:
thresholds_ls = np.arange(-0.8, 0.11, 0.1)

print(
    f"{'Threshold':>10} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds_ls:
    y_pred = np.where(
        val_scores_ls >= threshold,
        1,
        -1
    )

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:10.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

 Threshold   Accuracy  Precision     Recall         F1    Pred +1
     -0.80     67.26%     19.67%     87.77%     0.3213      25861
     -0.70     78.03%     25.31%     76.32%     0.3802      17472
     -0.60     84.92%     31.43%     59.88%     0.4122      11042
     -0.50     88.99%     38.60%     41.78%     0.4013       6272
     -0.40     90.81%     46.37%     25.76%     0.3312       3220
     -0.30     91.32%     53.27%     14.05%     0.2223       1528
     -0.20     91.39%     60.75%      7.02%     0.1259        670
     -0.10     91.31%     67.27%      3.19%     0.0610        275
     -0.00     91.25%     78.12%      1.29%     0.0255         96
      0.10     91.22%     91.67%      0.57%     0.0113         36


Both class-conditional mean scores are negative (-0.8505 and -0.5424), making a negative threshold relevant. The highest grid F1 is 0.4122 at threshold -0.60 (accuracy 84.92%). A threshold of 0 detects very few positives.


## 10. MSE Gradient Descent: Stability and Recovery

Use the same intercept design as Least Squares to compare the iterative solution with the closed-form reference. Start with 100 iterations and gamma = 0.1.


MSE Gradient Descent

In [50]:
from implementations import mean_squared_error_gd

initial_w_mse = np.zeros(x_tr_int.shape[1])

w_mse_gd, loss_mse_gd = mean_squared_error_gd(
    y_tr,
    x_tr_int,
    initial_w_mse,
    max_iters=100,
    gamma=0.1,
)

print("Training loss:", loss_mse_gd)
print("Least squares loss:", loss_ls)

print(
    "Weight distance to least squares:",
    np.linalg.norm(w_mse_gd - w_ls)
)

Training loss: 15198.980793298184
Least squares loss: 0.1362479199820299
Weight distance to least squares: 38.21137843249877


In [51]:
H = (x_tr_int.T @ x_tr_int) / x_tr_int.shape[0]

eigvals = np.linalg.eigvalsh(H)
lambda_max = eigvals[-1]

gamma_max = 2.0 / lambda_max

print("Largest Hessian eigenvalue:", lambda_max)
print("Approx. maximum stable gamma:", gamma_max)
print("Current gamma:", 0.1)

Largest Hessian eigenvalue: 21.09384167148154
Approx. maximum stable gamma: 0.09481440276021226
Current gamma: 0.1


The first run is unstable: MSE reaches 15,198.98. The largest Hessian eigenvalue is 21.09384, giving the quadratic-GD step-size bound 2/lambda_max = 0.094814; gamma = 0.1 exceeds it. Retry with gamma = 0.05 and a longer budget.


In [52]:
w_mse_gd_005, loss_mse_gd_005 = mean_squared_error_gd(
    y_tr,
    x_tr_int,
    np.zeros(x_tr_int.shape[1]),
    max_iters=500,
    gamma=0.05,
)

print("GD training loss:", loss_mse_gd_005)
print("Least squares loss:", loss_ls)

print(
    "Weight distance to least squares:",
    np.linalg.norm(w_mse_gd_005 - w_ls)
)

GD training loss: 0.13689749610011787
Least squares loss: 0.1362479199820299
Weight distance to least squares: 4.358570758009528


In [53]:
val_scores_gd = x_val_int @ w_mse_gd_005

thresholds_gd = np.arange(-0.8, 0.11, 0.1)

print(
    f"{'Threshold':>10} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds_gd:
    y_pred = np.where(
        val_scores_gd >= threshold,
        1,
        -1
    )

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:10.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

 Threshold   Accuracy  Precision     Recall         F1    Pred +1
     -0.80     66.63%     19.39%     88.06%     0.3179      26313
     -0.70     77.75%     24.95%     75.72%     0.3754      17584
     -0.60     85.02%     31.38%     58.64%     0.4088      10830
     -0.50     89.15%     38.79%     39.65%     0.3922       5924
     -0.40     90.92%     47.20%     23.81%     0.3166       2924
     -0.30     91.37%     55.11%     12.48%     0.2035       1312
     -0.20     91.39%     63.24%      5.94%     0.1085        544
     -0.10     91.29%     68.63%      2.42%     0.0467        204
     -0.00     91.24%     84.85%      0.97%     0.0191         66
      0.10     91.21%     96.55%      0.48%     0.0096         29


At 500 iterations, MSE falls to 0.136897, close to Least Squares, and best grid F1 is 0.4088 at threshold -0.60. Weight distance remains 4.3586, so similar objective values do not establish convergence to the same parameter vector.


## 11. MSE Stochastic Gradient Descent

Use single-sample updates with seed 42. The initial run tests gamma = 0.001 over 300,000 updates; inspect individual sample scales after observing instability.


MSE-SGD

In [54]:
from implementations import mean_squared_error_sgd

np.random.seed(42)

w_mse_sgd, loss_mse_sgd = mean_squared_error_sgd(
    y_tr,
    x_tr_int,
    np.zeros(x_tr_int.shape[1]),
    max_iters=300_000,
    gamma=0.001,
)

print("SGD training loss:", loss_mse_sgd)
print("Least squares loss:", loss_ls)

print(
    "Weight distance to least squares:",
    np.linalg.norm(w_mse_sgd - w_ls)
)

SGD training loss: 6.000139169311754e+189
Least squares loss: 0.1362479199820299
Weight distance to least squares: 9.626673161157742e+94


In [55]:
row_norm_sq = np.sum(x_tr_int ** 2, axis=1)

print("Squared row norm statistics:")
print(f"Min    : {np.min(row_norm_sq):.4f}")
print(f"Median : {np.median(row_norm_sq):.4f}")
print(f"95%    : {np.percentile(row_norm_sq, 95):.4f}")
print(f"99%    : {np.percentile(row_norm_sq, 99):.4f}")
print(f"Max    : {np.max(row_norm_sq):.4f}")

print(
    "\nApprox. conservative gamma from max row norm:",
    1.0 / np.max(row_norm_sq)
)

Squared row norm statistics:
Min    : 36.4799
Median : 181.0661
95%    : 825.6234
99%    : 2084.3039
Max    : 114712.4068

Approx. conservative gamma from max row norm: 8.717452874547948e-06


The initial loss explodes to approximately 6.0e189. Squared row norms range up to 114,712.41; their reciprocal suggests a conservative step-size scale of 8.72e-6. This is a stability diagnostic, not a convergence guarantee. Retry at 5e-6 for 1,000,000 updates.


In [56]:
np.random.seed(42)

w_mse_sgd_small, loss_mse_sgd_small = mean_squared_error_sgd(
    y_tr,
    x_tr_int,
    np.zeros(x_tr_int.shape[1]),
    max_iters=1_000_000,
    gamma=5e-6,
)

print("SGD training loss:", loss_mse_sgd_small)
print("Least squares loss:", loss_ls)

print(
    "Weight distance to least squares:",
    np.linalg.norm(w_mse_sgd_small - w_ls)
)

SGD training loss: 0.13738221463750394
Least squares loss: 0.1362479199820299
Weight distance to least squares: 4.3655312206444545


In [57]:
val_scores_sgd = x_val_int @ w_mse_sgd_small

thresholds_sgd = np.arange(-0.8, 0.11, 0.1)

print(
    f"{'Threshold':>10} "
    f"{'Accuracy':>10} "
    f"{'Precision':>10} "
    f"{'Recall':>10} "
    f"{'F1':>10} "
    f"{'Pred +1':>10}"
)

for threshold in thresholds_sgd:
    y_pred = np.where(
        val_scores_sgd >= threshold,
        1,
        -1
    )

    accuracy, precision, recall, f1 = evaluate_predictions(
        y_val, y_pred
    )

    print(
        f"{threshold:10.2f} "
        f"{accuracy:10.2%} "
        f"{precision:10.2%} "
        f"{recall:10.2%} "
        f"{f1:10.4f} "
        f"{np.sum(y_pred == 1):10d}"
    )

 Threshold   Accuracy  Precision     Recall         F1    Pred +1
     -0.80     66.10%     19.19%     88.39%     0.3153      26695
     -0.70     77.32%     24.58%     75.86%     0.3713      17882
     -0.60     84.81%     31.00%     58.74%     0.4058      10980
     -0.50     88.96%     37.96%     39.43%     0.3868       6019
     -0.40     90.77%     45.64%     23.55%     0.3107       2991
     -0.30     91.33%     53.78%     12.63%     0.2046       1361
     -0.20     91.30%     57.63%      5.54%     0.1011        557
     -0.10     91.25%     63.00%      2.17%     0.0420        200
     -0.00     91.21%     71.64%      0.83%     0.0164         67
      0.10     91.19%     83.33%      0.35%     0.0069         24


The smaller-step run yields MSE = 0.137382 and best grid F1 = 0.4058 at threshold -0.60. It is stable in this recorded run but remains slightly below Least Squares and batch GD in validation F1. One SGD update uses one sample, so update counts are not directly comparable with full-batch iterations.


## 12. Baseline Comparison Summary

All entries below come from saved outputs on the same validation split. Except for the majority and initial no-intercept logistic references, thresholds select the highest displayed F1 within each experiment's tested grid. Accuracy is reported at that threshold, not independently maximized. Regression thresholds act on linear scores; logistic thresholds act on sigmoid outputs, so their numeric values are not directly comparable.

| Experiment | Settings | Threshold | Accuracy | Positive-class F1 |
| --- | --- | ---: | ---: | ---: |
| Majority reference | Always -1 | — | 91.17% | 0.0000 |
| Ridge | No intercept; lambda = 0.01 | 0.25 | 86.49% | 0.4116 |
| Initial logistic | No intercept; 100 iterations; gamma = 0.1 | 0.50 | 63.46% | 0.3023 |
| Logistic with intercept | 100 iterations; gamma = 0.1 | 0.25 | 83.99% | 0.4023 |
| Logistic + missing indicators | 100 iterations; gamma = 0.1 | 0.20 | 86.93% | 0.4002 |
| Longer logistic run | 500 iterations; gamma = 0.1 | 0.20 | 86.20% | 0.4166 |
| Tuned logistic | 500 iterations; gamma = 0.2 | 0.20 | 86.68% | 0.4201 |
| Tuned logistic | 500 iterations; gamma = 0.5 | 0.20 | 86.80% | 0.4201 |
| Regularized logistic | 500 iterations; gamma = 0.5; lambda = 0.001 | 0.20 | 86.43% | 0.4215 |
| Least Squares | With intercept | -0.60 | 84.92% | 0.4122 |
| MSE-GD | With intercept; 500 iterations; gamma = 0.05 | -0.60 | 85.02% | 0.4088 |
| MSE-SGD | With intercept; 1,000,000 updates; gamma = 5e-6 | -0.60 | 84.81% | 0.4058 |

Regularized logistic has the highest recorded F1, 0.4215, only slightly above tuned logistic. Threshold choice, the intercept, and optimization stability all materially affect these baselines. Ridge lacks an intercept, missing indicators were not retested at the longer budget, and SGD uses a different update budget; these are not fully controlled comparisons. Repeated model and threshold selection on this validation set can make selected scores optimistic. No independent test performance or cross-validation results are recorded here.


## Feature Selection and Feature Engineering
